In [1]:
import os, subprocess, json, shlex
from pathlib import Path
from datetime import datetime
from typing import Tuple, Dict
from dotenv import load_dotenv
load_dotenv(Path("configs") / "local.env")

from src.minbpe import RegexTokenizer
#from src.gpt import GPTLanguageModel
from src.transformer.model_relative_positional_encoding import GPTLanguageModel

import matplotlib.pyplot as plt
import torch
torch.set_float32_matmul_precision('high')
from torch.utils.data import Dataset, DataLoader

In [2]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

def check_ckpt_files(ckpt_dir, match="fine-tuning_*.pt", keep_last=5):
    def handle(pt_file):
        json_file = str(pt_file).replace(".pt", ".json")
        with open(json_file, 'r') as f:
            d = json.load(f)
        return (d['epoch'], d['train_loss'], pt_file)

    records = [handle(f) for f in ckpt_dir.glob(match)]
    if len(records) <= 5:
        return

    ignore_epoches = []

    #key=lambda x: x.stat().st_ctime,
    ordered = sorted(records, key=lambda x: x[1], reverse=False)
    ignore_epoches.append(ordered[0][0])

    ordered = sorted(records, key=lambda x: x[0], reverse=True)
    for v in ordered[:5]:
        ignore_epoches.append(v[0])

    for e in records:
        if e[0] not in ignore_epoches:
            print(f"Remove ckpt file: {e[2]}")
            os.remove(e[2])

def send_notification(title, message):
    cmd = os.getenv("send_notification")
    if cmd is None:
        return

    command = shlex.split(cmd)
    command.append(title)
    command.append(message)

    _ = subprocess.Popen(command)

In [3]:
class FineTuningDataset(Dataset):
    def __init__(self, data: torch.Tensor, device: torch.device, padding_token: int):
        self.data = data
        self.device = device
        self.padding_token = padding_token

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        sample = self.data[index]
        x = sample.to(self.device)
        y = sample[1:].to(self.device)
        padding_tensor = torch.tensor([self.padding_token], device=self.device)
        y = torch.cat((y, padding_tensor))

        return x, y

@torch.no_grad()
def estimate_loss(
    model: torch.nn.Module,
    eval_dataset: Dict[str, DataLoader],
) -> Dict[str, float]:
    output = {}
    model.eval()

    for split, loader in eval_dataset.items():
        losses = []
        for x, y in loader:
            with torch.no_grad():
                _, loss = model(x, y)
            losses.append(loss.item())
        output[split] = sum(losses) / len(losses)

    model.train()
    return output

In [4]:
run_name = "ch09_fine-tuning"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer_dir = Path("data") / "tokenizer"

checkpoint_dir = Path("data") / "ch09"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [5]:
tokenizer = RegexTokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

train_tensor = torch.load(tokenizer_dir /'ch05_train.ft.pt')
val_tensor = torch.load(tokenizer_dir /'ch05_validation.ft.pt')

In [6]:
padding_token = -100
learning_rate = 1e-4
batch_size = 96
epoch_last = 0
num_epoches = 500

train_dataset = FineTuningDataset(data=train_tensor, device=device, padding_token=padding_token)
val_dataset = FineTuningDataset(data=val_tensor, device=device, padding_token=padding_token)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size)
print(f"Dataset: train={len(train_loader)}, validation={len(val_loader)}")

Dataset: train=329, validation=17


In [9]:
from_ft_model = False

ckpt_files = sorted(
    checkpoint_dir.glob("fine-tuning_*.pt"),
    #key=lambda x: x.stat().st_ctime,
    key=lambda x: int(x.name.replace("fine-tuning_", "").replace(".pt", "")),
    reverse=True,
)

if len(ckpt_files) > 0:
    checkpoint_path = ckpt_files[0]
    tag = checkpoint_path.name.replace("fine-tuning_", "").replace(".pt", "")
    epoch_last = int(tag)
    from_ft_model = True
else:
    checkpoint_path = Path("data") / "ch09" / "checkpoint_005-629785.pt"

checkpoint = torch.load(checkpoint_path, weights_only=True, map_location=device)
parameters = checkpoint['meta']['parameters']
print(f"Loaded checkpoint: path={checkpoint_path}, from_ft_model={from_ft_model}")
print(f"Model parameters: {parameters}")

Loaded checkpoint: path=data/ch09/fine-tuning_000207.pt, from_ft_model=True
Model parameters: {'vocab_size': 1029, 'n_embd': 512, 'block_size': 256, 'n_head': 8, 'n_layer': 4, 'dropout': 0.2}


In [10]:
model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    block_size=parameters['block_size'],
    n_embd=parameters['n_embd'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    ignore_index=tokenizer.special_tokens["<|padding|>"],
    device=device,
)

model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.load_state_dict(checkpoint['model_state_dict'])
if from_ft_model:
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

num_parameters = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Fine-tuning model: num_parameters={num_parameters:.3f}M, learning_rate={optimizer.param_groups[0]['lr']:.6f}")
# print_model_structure(model)

Fine-tuning model: num_parameters=13.660M, learning_rate=0.000100


In [ ]:
#tokens = torch.tensor([
#    [1, 23, 456, 789],
#    [2, 34, 567, 890],
#])

#te = model.token_embedding_table(tokens)
#pe = model.token_embedding_table(tokens)

#b0o = model.blocks[0](b0i)
#b1o = model.blocks[0](b0o)

#input_tokens = tokenizer.encode("hello, world", allowed_special="all")
#input_tokens = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

#model.eval()

#with torch.no_grad():
#    output = model.generate(input_tokens=input_tokens, max_new_tokens=100)
#    print(tokenizer.decode(output[0].tolist()))

In [ ]:
train_losses, val_losses = [], []
total_steps = len(train_loader) * num_epoches

print(f"Start training: epoch_last={epoch_last}, num_epoches={num_epoches}, total_steps={total_steps:_}")

def _estimate_and_save(epoch, notify=False):
    t0 = datetime.now()

    checkpoint_prefix = str(checkpoint_dir / f"fine-tuning_{epoch:06}")

    losses = estimate_loss(
        model=model,
        eval_dataset={'train': train_loader, 'val': val_loader},
    )

    train_losses.append(losses['train'])
    val_losses.append(losses['val'])

    meta = {
        'created_at': now(),
        'parameters': parameters,
        'epoch': epoch,
        'run_name': run_name,
        'batch_size': batch_size,
        'total_batches': len(train_loader),
        'learing_rate': optimizer.param_groups[0]['lr'],
        'train_loss': float(losses['train']),
        'val_loss': float(losses['val']),
    }

    checkpoint = {
        'meta': meta,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }

    with open(checkpoint_prefix + ".json", 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
        f.write("\n")

    torch.save(checkpoint, checkpoint_prefix+".pt")

    elapsed = datetime.now() - t0

    print(
        f"{now()} epoch={epoch}/{num_epoches}, train_loss={losses['train']:.3f}, validation_loss={losses['val']:.3f}, "
        f"elapsed={str(elapsed)}, checkpoint={checkpoint_prefix}.pt"
    )

    check_ckpt_files(checkpoint_dir)

    if notify:
        msg = f"epoch={epoch}/{num_epoches}, train_loss={losses['train']:.3f}, validation_loss={losses['val']:.3f}"
        send_notification(run_name, msg)

for epoch in range(epoch_last+1, num_epoches+1):
    for _, (x_batch, y_batch) in enumerate(train_loader):
        optimizer.zero_grad(set_to_none=True)
        logits, loss = model(x_batch, y_batch)
        loss.backward()
        optimizer.step()
        #batch_loss = loss.item()

    _estimate_and_save(epoch, notify=epoch % 10 == 0)

_estimate_and_save(epoch, notify=True)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Step")
plt.ylabel("Loss")
plt.title("Training and Validation Losses Over Steps")
plt.legend()
plt.grid()
plt.show()